# Chotu tokenizer · How text becomes the numbers the model reads

The model never sees letters. It sees **token ids**: whole numbers from 0 to 49,151. The **tokenizer** is the fixed dictionary and set of rules that turns text into those ids and back. We use SmolLM2's tokenizer as-is (a frozen decision), so everything the model learns, it learns *in these pieces*. Knowing the pieces explains a lot of model behaviour later: why arithmetic is hard or easy, why a rare name comes out misspelled, why " The" and "The" are different things to the model.

Everything below uses **live objects**: the real SmolLM2 tokenizer, our token shard `train_000.bin` from `magnateadi/chotu-fineweb-edu-10bt-tokens` (300M tokens), about 14,000 original FineWeb-Edu documents, our own test prompts, and one real chat-tuning conversation from `smol-smoltalk`.

| Part | Question it answers |
|---|---|
| 1 | What exactly is this tokenizer? |
| 2 | How do bytes become symbols (the Ġ and Ċ mystery)? |
| 3 | What does real text look like once it's tokenized? |
| 4 | How is one token built, merge by merge? |
| 5 | What's in the 49,152-token vocabulary? |
| 6 | Which tokens does our data actually use? |
| 7 | How are words split into tokens? |
| 8 | What happens to numbers? |
| 9 | Do capitals and spaces matter? |
| 10 | How well does it compress different documents? |
| 11 | How does it compare to GPT-2's tokenizer? |
| 12 | Is it lossless, and what's actually in the shards? |
| 13 | What does a chat look like to the model? |
| 14 | What this means for Chotu |

In [ ]:
# 0. Setup: tokenizer, our cached shard (shared with eda_data.ipynb), and a sample of source documents
%matplotlib inline
import sys, json, re, collections
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

REPO = next(p for p in [Path.cwd(), Path.cwd().parent, Path.home() / 'Chotu', Path.home() / 'apps' / 'Chotu'] if (p / 'chotu' / 'config.py').exists())
sys.path.insert(0, str(REPO))
from dotenv import load_dotenv; load_dotenv(REPO / '.env')
from huggingface_hub import hf_hub_download, HfFileSystem
import pyarrow.parquet as pq
from transformers import AutoTokenizer
from chotu.config import TOKENIZER_ID
from chotu import xray as X

DATASET = 'magnateadi/chotu-fineweb-edu-10bt-tokens'
RAW = 'HuggingFaceFW/fineweb-edu'
CACHE = Path.home() / 'chotu-data' / 'eda'; CACHE.mkdir(parents=True, exist_ok=True)
train_path = Path(hf_hub_download(DATASET, 'shards/train_000.bin', repo_type='dataset', local_dir=CACHE))

tok = AutoTokenizer.from_pretrained(TOKENIZER_ID); tok.model_max_length = 10**9
gpt2 = AutoTokenizer.from_pretrained('openai-community/gpt2'); gpt2.model_max_length = 10**9
TJ = json.loads(tok.backend_tokenizer.to_str())          # the tokenizer's full definition
VOCAB = TJ['model']['vocab']                              # piece -> id
MERGES = [tuple(m) if isinstance(m, list) else tuple(m.split(' ')) for m in TJ['model']['merges']]
RANK = {m: i for i, m in enumerate(MERGES)}
N = len(tok)
id2piece = {i: p for p, i in VOCAB.items()}
for t, i in tok.get_added_vocab().items(): id2piece[i] = t

# Byte-level BPE writes every byte as a printable character (space -> Ġ, newline -> Ċ). This is that table.
def bytes_to_unicode():
    bs = list(range(ord('!'), ord('~') + 1)) + list(range(ord('¡'), ord('¬') + 1)) + list(range(ord('®'), ord('ÿ') + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = bytes_to_unicode(); U2B = {v: k for k, v in B2U.items()}
def piece_bytes(i):
    p = id2piece[i]
    return p.encode() if i < 17 else bytes(U2B[c] for c in p)
def show_piece(i):
    return piece_bytes(i).decode('utf-8', errors='replace').replace(' ', '␣').replace('\n', '↵').replace('\t', '⇥')

PALETTE = ['#fde68a', '#bfdbfe', '#bbf7d0', '#fecaca', '#ddd6fe', '#fed7aa', '#a5f3fc', '#fbcfe8']
def chips(ids, note=None, mark=None):
    out = []
    for k, i in enumerate(ids):
        bg = '#ef4444' if mark and mark(k, i) else PALETTE[k % len(PALETTE)]
        fg = 'white' if mark and mark(k, i) else '#111'
        out.append(f"<span style='display:inline-block;margin:2px;padding:2px 5px;border-radius:4px;background:{bg};color:{fg};"
                   f"font-family:monospace;text-align:center'>{show_piece(i).replace('<', '&lt;')}<br><small style='opacity:.65'>{i}</small></span>")
    head = f"<div style='font-family:sans-serif;margin:6px 0 2px'>{note}</div>" if note else ''
    return head + '<div>' + ''.join(out) + '</div>'

# ~14,000 original documents: one random 1,000-row block from each of the 14 source files
fs, rng, parts = HfFileSystem(), np.random.default_rng(1), []
for i in range(14):
    pf = pq.ParquetFile(fs.open(f'datasets/{RAW}/sample/10BT/{i:03d}_00000.parquet', 'rb'))
    parts.append(pf.read_row_group(int(rng.choice(pf.num_row_groups)), columns=['id', 'text', 'url', 'score', 'language_score', 'token_count']).to_pandas())
raw = pd.concat(parts, ignore_index=True)
raw['domain'] = raw.url.str.extract(r'https?://(?:www\.)?([^/]+)')[0]
enc = tok(raw.text.tolist(), add_special_tokens=False)['input_ids']
raw['n_tok'] = [len(e) for e in enc]
raw['n_byte'] = raw.text.map(lambda t: len(t.encode('utf-8')))
raw['bpt'] = raw.n_byte / raw.n_tok

# Token counts from 300M real tokens of our shard
counts, total = X.token_frequency([train_path], max_tokens=300_000_000)
counts = counts[:N]
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
print(f'tokenizer: {TOKENIZER_ID} | {N:,} ids | {len(MERGES):,} merges')
print(f'{len(raw):,} source documents ({raw.n_tok.sum():,} tokens) | {total:,} shard tokens from {train_path.name}')

## 1 · What exactly is this tokenizer?

It's a **byte-level BPE** tokenizer (*BPE* = byte-pair encoding). It starts from single **bytes** (235 of the 256 possible bytes have a token), so almost any text can be written, even emoji or Chinese. The 21 missing bytes are rare control characters and bytes that can't occur in valid text. Then it applies a list of **merges** learned from a big text corpus: "join `t` and `h` into `th`", "join `Ġ` and `the` into `Ġthe`", and so on. Each merge adds one token to the vocabulary.

Two settings matter a lot:
- **No normalizer**: the text is never changed before tokenizing. Capitals, accents and spacing are kept exactly.
- **17 special tokens** at ids 0 to 16. Only three matter to us: `<|endoftext|>` (0) separates documents, and `<|im_start|>` (1) / `<|im_end|>` (2) mark chat turns. The rest come from SmolLM2's code training.

**Look for:** how much of the vocabulary is raw bytes versus learned merges, and how often each special token shows up in our pretraining data.

In [ ]:
n_special = 17
single = [i for i in range(n_special, N) if len(id2piece[i]) == 1]
rows = {
    'algorithm': TJ['model']['type'] + ' (byte-level)',
    'vocabulary size': f'{N:,}',
    'special tokens (ids 0-16)': n_special,
    'single-byte tokens': f'{len(single):,}',
    'merged tokens (learned)': f'{N - n_special - len(single):,}',
    'merge rules': f'{len(MERGES):,}',
    'normalizer (changes text before tokenizing)': TJ['normalizer'] or 'none: text is used exactly as written',
    'pre-tokenizer (first split into words)': f"{TJ['pre_tokenizer']['type']}, regex split, add_prefix_space={TJ['pre_tokenizer']['add_prefix_space']}",
    'built-in chat template': 'yes' if getattr(tok, 'chat_template', None) else 'none: we write ChatML ourselves (frozen decision)',
}
display(pd.DataFrame({'value': rows}))

fig, ax = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={'width_ratios': [3, 2]})
parts_ = [('special', n_special, '#ef4444'), ('single bytes', len(single), '#f59e0b'), ('learned merges', N - n_special - len(single), '#60a5fa')]
left = 0
for name, n, c in parts_:
    ax[0].barh(0, n, left=left, color=c, label=f'{name}: {n:,}'); left += n
ax[0].set(yticks=[], xlim=(0, N), title='The 49,152 ids, by where they come from', xlabel='token id'); ax[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)
sp = pd.Series({id2piece[i]: int(counts[i]) for i in range(n_special)})
ax[1].barh(sp.index[::-1], sp.values[::-1] + 1, color=['#ef4444' if v > 0 else '#d1d5db' for v in sp.values[::-1]])
ax[1].set_xscale('log'); ax[1].set(title=f'Special tokens in {total/1e6:.0f}M shard tokens (log scale, +1)'); ax[1].tick_params(axis='y', labelsize=8)
plt.tight_layout(); plt.show()
print(f'<|endoftext|> appears {counts[0]:,} times: once per document.')
for i in range(1, n_special):
    if counts[i]: print(f'surprise: {id2piece[i]} (id {i}) appears {counts[i]:,} times. The literal text "{id2piece[i]}" in a web page was turned into the special token.')
print(f'chat markers: <|im_start|> {counts[1]:,} | <|im_end|> {counts[2]:,}')
missing = [b for b in range(256) if B2U[b] not in VOCAB]
print(f'{256 - len(missing)} of 256 bytes have a token; missing: ' + ' '.join(f'{b:02x}' for b in missing))

**Think about it**
1. Why start from 256 *bytes* instead of a list of letters? What would happen to an emoji or a Hindi word if we started from English letters only?
2. There's no normalizer, so "Hello" and "hello" are tokenized differently. What's the upside of never changing the text? What's the cost?
3. The chat markers 1 and 2 appear zero times in pretraining. What will their embeddings look like when chat tuning starts?
4. A web page that literally contains the text `<filename>` became special token 6 in our shard. Harmless at a handful of times, but what if a page contained `<|im_start|>`?

## 2 · Bytes become symbols: the Ġ and Ċ mystery

Computers store text as **bytes**. English letters are one byte each, but `é` is two bytes and many symbols are three or four. Byte-level BPE gives each of the 256 bytes a printable stand-in character, so that tokens can be stored as ordinary strings. That's why you'll see `Ġ` for a **space** and `Ċ` for a **newline** inside tokens: `Ġthe` means " the".

Below are the most common non-English characters in our sampled documents, with their bytes, their stand-in characters, and the tokens they become.

In [ ]:
char_counts = collections.Counter(c for t in raw.text.iloc[:4000] for c in t if ord(c) > 127)
rows = []
for c, n in char_counts.most_common(14):
    ids = tok(c, add_special_tokens=False)['input_ids']
    rows.append({'character': c, 'name': __import__('unicodedata').name(c, '?').title(), 'times in 4,000 docs': n,
                 'UTF-8 bytes': ' '.join(f'{b:02x}' for b in c.encode()),
                 'stand-in characters': ''.join(B2U[b] for b in c.encode()),
                 'token ids': ids, 'tokens': len(ids)})
display(pd.DataFrame(rows))
basic = [(' ', 'space'), ('\n', 'newline'), ('\t', 'tab')]
display(Markdown('Invisible characters: ' + ' · '.join(f'**{name}** → `{B2U[ord(ch)]}`' for ch, name in basic)))

**Think about it**
1. `é` and `–` are common enough to get their own token, while rarer characters take two or three tokens. What decides which ones got a single token?
2. If a token holds only *half* of a character's bytes, what would the model print if it stopped right there?

## 3 · What real text looks like once it's tokenized

Here are real snippets from our sampled documents, cut into their actual tokens. Each coloured chip is one token, with its id underneath. `␣` is a space and `↵` a newline. The snippets were picked automatically: a typical document, the most number-heavy one, the most code-like one, the one least likely to be English, and one with a web address.

**Look for:** common words are one chip each and usually carry their leading space. Rare words, numbers, code and other languages break into many small chips.

In [ ]:
def first_chars(t, n=320): return t[:n].rsplit(' ', 1)[0]
digit_share = raw.text.map(lambda t: sum(c.isdigit() for c in t[:2000]) / max(1, len(t[:2000])))
codey = raw.text.str.count(r'[{};=()]') / raw.text.str.len()
picks = {
    'Typical document (median quality score)': raw.iloc[(raw.score - raw.score.median()).abs().argmin()].text,
    'Most number-heavy': raw.text.iloc[digit_share.idxmax()],
    'Most code-like': raw.text.iloc[codey.idxmax()],
    'Least likely to be English': raw.text.iloc[raw.language_score.idxmin()],
}
url_doc = raw.text[raw.text.str.contains(r'https?://\S{15,}')].iloc[0]
m = re.search(r'https?://\S{15,}', url_doc)
picks['A web address'] = url_doc[max(0, m.start() - 60): m.end() + 40]
for label, text in picks.items():
    snippet = first_chars(text)
    display(HTML(f"<h4 style='font-family:sans-serif;margin:16px 0 4px'>{label}</h4>" + X.token_lens(tok, snippet)))

**Think about it**
1. In the typical document, roughly how many characters does one chip hold? Is that what you expected?
2. The web address breaks into many small pieces. Why might a model find URLs hard to reproduce exactly?
3. Find a word that's split into two or more chips. Can you guess why it wasn't a single token?

## 4 · How one token is built, merge by merge

BPE tokenizes a word by starting from single bytes and repeatedly applying the **earliest-learned merge** that fits, until no merge applies. The merge's position in the list (its *rank*) says how early it was learned, and early merges are the most common pairs in SmolLM2's training text.

Below, we replay this by hand using the tokenizer's real merge list. The words come from our data: the most frequent long word that ends up as **one** token, and the most frequent word that stays in **several** pieces. The project's own name is included too. The last line checks our replay against the real tokenizer.

In [ ]:
pretok = tok.backend_tokenizer.pre_tokenizer.pre_tokenize_str
def bpe_steps(word):
    parts, steps = list(word), []
    steps.append((parts[:], None))
    while len(parts) > 1:
        r, k = min((RANK.get((a, b), 10**9), k) for k, (a, b) in enumerate(zip(parts, parts[1:])))
        if r == 10**9: break
        pair, new, j = MERGES[r], [], 0
        while j < len(parts):
            if j < len(parts) - 1 and (parts[j], parts[j + 1]) == pair: new.append(parts[j] + parts[j + 1]); j += 2
            else: new.append(parts[j]); j += 1
        parts = new; steps.append((parts[:], r))
    return steps

word_counts = collections.Counter(w for t in raw.text.iloc[:3000] for w, _ in pretok(t))
uniq = [w for w, _ in word_counts.most_common(60_000)]
wtext = [bytes(U2B[c] for c in w).decode('utf-8', errors='replace') for w in uniq]
wtok = tok(wtext, add_special_tokens=False)['input_ids']
ntok = dict(zip(uniq, map(len, wtok)))
one_tok_long = next(w for w in uniq if ntok[w] == 1 and len(w) >= 11 and w[1:].isalpha())
multi = next(w for w in uniq if ntok[w] >= 3 and w[1:].isalpha())
demo = [one_tok_long, multi, 'ĠChotu']
for w in demo:
    steps = bpe_steps(w)
    rows = ''.join(f"<tr><td style='padding:2px 10px;color:#666;font-family:monospace'>{'start' if r is None else f'merge #{r:,}'}</td><td>"
                   + chips([VOCAB[p] for p in parts]) + '</td></tr>' for parts, r in steps)
    real = tok(bytes(U2B[c] for c in w).decode(), add_special_tokens=False)['input_ids']
    ok = [VOCAB[p] for p in steps[-1][0]] == real
    display(HTML(f"<h4 style='font-family:sans-serif;margin:14px 0 4px'>“{w.replace('Ġ', ' ')}”: {len(steps)-1} merges → {len(steps[-1][0])} token(s) "
                 f"{'✓ matches the real tokenizer' if ok else '✗ differs from the real tokenizer'}</h4><table>{rows}</table>"))

### Does our data agree with SmolLM2's merge order?

A merged token's id is exactly its merge rank plus 252 (17 special + 235 single-byte ids come first). So a low id means the pair was very common in SmolLM2's training text. If FineWeb-Edu looks like that text, low ids should also be the most frequent tokens *in our shard*. Each dot below is one token.

In [ ]:
ids = np.arange(n_special, N); c = counts[n_special:N]
used = c > 0
fig, ax = plt.subplots(figsize=(15, 4))
hb = ax.hexbin(ids[used], c[used], gridsize=(80, 30), yscale='log', bins='log', cmap='viridis', mincnt=1)
ax.set(xlabel='token id (low = merge learned early)', ylabel='times seen in our 300M tokens (log)', title='Token id vs how often our data uses it')
plt.colorbar(hb, ax=ax, label='tokens per cell (log)'); plt.show()
rho = pd.Series(ids[used]).rank().corr(pd.Series(c[used]).rank())
print(f'rank correlation between id and frequency: {rho:.2f} (−1 = perfectly "lower id, more frequent")')

**Think about it**
1. The long word became a single token after many merges, but the other word didn't. What does that tell you about how often each appeared in SmolLM2's training text?
2. "Chotu" is split into pieces. Will the model still be able to learn the word? What makes it harder than learning " the"?
3. The cloud of dots slopes downward but is wide. What kinds of tokens might be common in SmolLM2's data (which included code) but rare in ours?

## 5 · The vocabulary map: what's in the 49,152 tokens?

We sort every token into a kind:
- **word start**: begins with a space and then letters, like `␣the`, `␣photosynthesis`
- **word piece**: letters with no space, so it continues a word, like `ing`, `tion`
- **digit**: a single digit (this tokenizer never joins digits)
- **whitespace**: spaces, newlines and tabs only, like `␣␣␣` (code indentation) or `↵↵`
- **non-ASCII**: contains bytes outside plain English, like accents, symbols, other scripts, or part of a character
- **punctuation and symbols**: everything else, like `.`, `),`, `://`

**Look for:** a kind can take a big share of the dictionary but little of the text, or the other way round.

In [ ]:
def kind(i):
    if i < n_special: return 'special'
    b = piece_bytes(i)
    if any(x > 127 for x in b): return 'non-ASCII'
    s = b.decode()
    if s.strip() == '': return 'whitespace'
    if s.isdigit(): return 'digit'
    if s[0] == ' ' and s[1:].isalpha(): return 'word start'
    if s.isalpha(): return 'word piece'
    return 'punctuation & symbols'
vocab = pd.DataFrame({'id': np.arange(N)})
vocab['kind'] = vocab.id.map(kind)
vocab['text'] = vocab.id.map(show_piece)
vocab['bytes'] = vocab.id.map(lambda i: len(piece_bytes(i)))
vocab['count'] = counts
order = ['word start', 'word piece', 'punctuation & symbols', 'whitespace', 'digit', 'non-ASCII', 'special']
g = vocab.groupby('kind').agg(in_vocab=('id', 'size'), in_text=('count', 'sum')).reindex(order)
g['% of vocabulary'] = g.in_vocab / N * 100; g['% of our tokens'] = g.in_text / total * 100

fig, ax = plt.subplots(1, 2, figsize=(16, 4.2))
y = np.arange(len(order))
ax[0].barh(y + 0.2, g['% of vocabulary'], 0.4, label='share of vocabulary', color='#9ca3af')
ax[0].barh(y - 0.2, g['% of our tokens'], 0.4, label='share of our text', color='#60a5fa')
ax[0].set_yticks(y, order); ax[0].invert_yaxis(); ax[0].legend(); ax[0].set(title='Kinds of token: dictionary vs real usage', xlabel='%')
bins = np.arange(1, 18)
ax[1].hist(vocab.bytes.clip(upper=16), bins=bins, alpha=.6, label='vocabulary (each token once)', density=True, color='#9ca3af')
ax[1].hist(vocab.bytes.clip(upper=16), bins=bins, weights=vocab['count'], alpha=.6, label='our text (weighted by use)', density=True, color='#60a5fa')
ax[1].set(title='Token length in bytes (16 = 16 or more)', xlabel='bytes per token'); ax[1].legend()
plt.tight_layout(); plt.show()
display(g[['in_vocab', '% of vocabulary', '% of our tokens']].round(2))

longest = vocab[vocab.id >= n_special].sort_values('bytes', ascending=False).head(15)
display(HTML('<b style="font-family:sans-serif">The 15 longest tokens in the vocabulary, and how often our data uses them</b>'))
display(longest[['id', 'text', 'bytes', 'kind', 'count']].reset_index(drop=True))

**Think about it**
1. Word starts take a big share of both the dictionary and the text. Why does it help that the space is glued to the *front* of a word rather than the end?
2. Many of the longest tokens are runs of spaces or symbols, or come from code. Why would a tokenizer trained partly on code learn `␣␣␣␣␣␣␣␣` as one token?
3. Most tokens in real text are short, even though the vocabulary has long ones. Why is that?

## 6 · Which tokens does our data actually use?

The vocabulary is the menu; this is what actually gets ordered. Below are the most-used word-start tokens and word pieces in our shard, then the tokens our data **never** uses in 300M tokens.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 6.2))
for a, k, col in [(ax[0], 'word start', '#60a5fa'), (ax[1], 'word piece', '#10b981'), (ax[2], 'punctuation & symbols', '#f59e0b')]:
    top = vocab[vocab.kind == k].nlargest(25, 'count')[::-1]
    a.barh(top.text, top['count'] / total * 100, color=col); a.set(title=f'Top 25 {k} tokens', xlabel='% of all tokens'); a.tick_params(axis='y', labelsize=9)
plt.tight_layout(); plt.show()

never = vocab[(vocab.id >= n_special) & (vocab['count'] == 0)]
rare = vocab[(vocab.id >= n_special) & (vocab['count'].between(1, 9))]
cover = np.cumsum(np.sort(counts)[::-1]) / total
print(f'used at least once: {(counts > 0).sum():,} of {N:,} | never used: {len(never):,} | used 1-9 times: {len(rare):,}')
print(f'the top 1,000 tokens cover {cover[999]:.0%} of all text; the top 10,000 cover {cover[9999]:.0%}')
display(never.kind.value_counts().rename('never-used tokens by kind').to_frame().T)
display(HTML(chips(never.sample(48, random_state=0).id.tolist(), note='<b>A random 48 of the never-used tokens</b> (mostly code, other languages and odd fragments)')))

**Think about it**
1. The never-used tokens still have embeddings, and weight decay shrinks them slowly towards zero. If a user types one at chat time, what might the model do?
2. The top 1,000 tokens cover most of the text. Does that mean the other 48,000 don't matter? Think about where facts and names live.

## 7 · Words into tokens

Most English words are a single token. Long, rare or technical words get split. **Tokens per word** (sometimes called *fertility*) measures how much the tokenizer has to chop. Below we split every word in 3,000 documents, both by unique word and weighted by how often each word appears.

In [ ]:
wf = pd.DataFrame({'word': uniq, 'tokens': [ntok[w] for w in uniq], 'freq': [word_counts[w] for w in uniq]})
wf['text'] = wf.word.map(lambda w: bytes(U2B[c] for c in w).decode('utf-8', errors='replace'))
wf['letters'] = wf.text.str.strip().str.len()
alpha = wf[wf.text.str.strip().str.isalpha()]
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
b = np.arange(1, 9)
ax[0].hist(alpha.tokens.clip(upper=7), bins=b, density=True, alpha=.6, color='#9ca3af', label='each unique word once')
ax[0].hist(alpha.tokens.clip(upper=7), bins=b, weights=alpha.freq, density=True, alpha=.6, color='#60a5fa', label='weighted by how often it appears')
ax[0].set(title='Tokens per word (7 = 7 or more)', xlabel='tokens'); ax[0].legend()
m = alpha.groupby(alpha.letters.clip(upper=20)).tokens.mean()
ax[1].plot(m.index, m.values, marker='o'); ax[1].set(title='Longer words need more tokens', xlabel='letters in word (20 = 20+)', ylabel='average tokens')
plt.tight_layout(); plt.show()
print(f'average tokens per word, by appearance: {np.average(alpha.tokens, weights=alpha.freq):.2f} | across unique words: {alpha.tokens.mean():.2f}')
display(HTML('<b style="font-family:sans-serif">The most common words that need 3 or more tokens</b>'))
display(HTML(''.join(chips(tok(r.text, add_special_tokens=False)['input_ids'], note=f'“{r.text.strip()}”, seen {r.freq:,}×') for r in alpha[alpha.tokens >= 3].head(10).itertuples())))

**Think about it**
1. Weighted by appearance, the average is close to one token per word. Unique words need more. What does that tell you about where the "hard" words are?
2. A word split into 3 pieces has to be predicted 3 times in a row. Why could that make rare names more likely to come out misspelled?

## 8 · Numbers

This tokenizer **splits every number into single digits**, so `2024` is four tokens: `2` `0` `2` `4`. GPT-2's tokenizer, by contrast, chunks numbers in uneven, frequency-based pieces. One digit per token means the model always sees the same "digit in a column" structure, which is known to help with arithmetic, at the cost of more tokens. The numbers below are real ones from our documents.

In [ ]:
nums = collections.Counter(m for t in raw.text.iloc[:5000] for m in re.findall(r'\d[\d,.]*\d|\d', t))
examples = []
for pat in [r'^(19|20)\d\d$', r'^\d+\.\d{2,}$', r'^\d{1,3}(,\d{3})+$', r'^\d{5,}$']:
    examples.append(next(n for n, _ in nums.most_common() if re.match(pat, n)))
for n in examples:
    display(HTML(f"<div style='display:flex;gap:28px;flex-wrap:wrap;align-items:flex-start'><div>{chips(tok(n, add_special_tokens=False)['input_ids'], note=f'<b>Chotu</b>: “{n}”')}</div>"
                 f"<div>{X.token_lens(gpt2, n).replace('tokens</b>', 'tokens</b> (GPT-2)')}</div></div>"))
lens = pd.Series({n: len(re.sub(r'\D', '', n)) for n in nums})
w = pd.Series(nums)
fig, ax = plt.subplots(figsize=(15, 3.2))
d = lens.groupby(lens.clip(upper=12)).apply(lambda s: w[s.index].sum())
ax.bar(d.index, d.values / d.values.sum() * 100, color='#f59e0b'); ax.set(title='How many digits do numbers in our documents have? (12 = 12+)', xlabel='digits (= tokens for Chotu)', ylabel='% of numbers')
plt.show()
print(f"digits are {vocab.loc[vocab.kind == 'digit', 'count'].sum() / total:.2%} of all our tokens")

**Think about it**
1. With one digit per token, how does the model know whether a `5` means five, fifty or five hundred? (Hint: what else is in the window?)
2. GPT-2 might write `2024` as one token in one place and two in another. Why would that make learning to add harder?

## 9 · Capitals and spaces matter

With no normalizer, the same word can be several different tokens depending on its capitals and whether a space comes first. To the model these are **unrelated ids** at the start of training: it has to learn from data that ` the` and ` The` mean the same thing. The table uses the most common word-start tokens in our data.

In [ ]:
rows = []
for w in [s.strip() for s in vocab[vocab.kind == 'word start'].nlargest(8, 'count').text.str.replace('␣', ' ')]:
    for form in [w, ' ' + w, w.capitalize(), ' ' + w.capitalize(), ' ' + w.upper()]:
        ids_ = tok(form, add_special_tokens=False)['input_ids']
        rows.append({'word': w, 'written as': repr(form), 'token ids': ids_, 'tokens': len(ids_),
                     'count in our data': int(counts[ids_[0]]) if len(ids_) == 1 else None})
t9 = pd.DataFrame(rows).drop_duplicates('written as')
display(t9.set_index(['word', 'written as']))
ws = vocab[vocab.kind == 'whitespace'].nlargest(10, 'count')
fig, ax = plt.subplots(figsize=(15, 3)); ax.bar(ws.text, ws['count'] / total * 100, color='#a78bfa')
ax.set(title='Most-used whitespace tokens (␣ = space, ↵ = newline)', ylabel='% of all tokens'); plt.show()

**Think about it**
1. ` the` is extremely common and ` THE` is rare. Will the model understand ` THE` as well? Where would it learn the connection?
2. A paragraph break (`↵↵`) is a single token. What does the model learn when it predicts a paragraph break?

## 10 · Compression: how much text fits in one token?

**Bytes per token** says how much text one token carries. More bytes per token means more text fits in the 2,048-token window and each training step covers more text. Plain English prose compresses well. Tables, code, maths and other languages compress badly. We measure every sampled document.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].hist(raw.bpt.clip(1, 7), bins=60, color='#60a5fa'); ax[0].axvline(raw.n_byte.sum() / raw.n_tok.sum(), color='r', ls='--')
ax[0].set(title='Bytes per token, per document', xlabel='bytes per token')
ax[1].hexbin(raw.language_score, raw.bpt.clip(1, 7), gridsize=40, bins='log', cmap='viridis', mincnt=1)
ax[1].set(title='Less English-looking → worse compression', xlabel='confidence the text is English', ylabel='bytes per token')
ax[2].hexbin(raw.score, raw.bpt.clip(1, 7), gridsize=40, bins='log', cmap='viridis', mincnt=1)
ax[2].set(title='Educational score vs compression', xlabel='educational score', ylabel='bytes per token')
plt.tight_layout(); plt.show()
print(f'overall: {raw.n_byte.sum() / raw.n_tok.sum():.2f} bytes per token | median document {raw.bpt.median():.2f} | worst 1% below {raw.bpt.quantile(.01):.2f}')
display(HTML('<b style="font-family:sans-serif">The three worst-compressed documents (first 200 characters)</b>'))
for r in raw[raw.n_tok > 200].nsmallest(3, 'bpt').itertuples():
    display(HTML(f"<div style='font-family:sans-serif;margin-top:10px'>{r.domain} · {r.bpt:.2f} bytes/token</div>" + X.token_lens(tok, r.text[:200])))

**Think about it**
1. A document at 2 bytes per token uses twice as many training tokens for the same amount of text as one at 4. Is the model spending its training budget wisely on those documents?
2. We judge the model in bits per byte, not loss per token. Using the chart, explain why per-token loss would be unfair when comparing two tokenizers.

## 11 · Compared with GPT-2's tokenizer

GPT-2 (our benchmark opponent) uses a 50,257-token byte-level BPE trained on different web text. FineWeb-Edu stores each document's GPT-2 token count, so we can compare on all 14,000 documents, and look at the same text side by side.

In [ ]:
ratio = raw.n_tok / raw.token_count
shared = set(VOCAB) & set(gpt2.get_vocab())
fig, ax = plt.subplots(figsize=(15, 3.4))
ax.hist(ratio.clip(0.7, 1.5), bins=80, color='#10b981'); ax.axvline(1, color='k', ls='--', lw=1)
ax.set(title='Chotu tokens ÷ GPT-2 tokens, per document (below 1 = Chotu uses fewer)', xlabel='ratio'); plt.show()
print(f'overall: Chotu uses {raw.n_tok.sum() / raw.token_count.sum():.3f}× as many tokens as GPT-2 | Chotu fewer on {(ratio < 1).mean():.0%} of documents')
print(f'shared vocabulary pieces: {len(shared):,} ({len(shared) / N:.0%} of ours)')
text = first_chars(raw.text.iloc[digit_share.sort_values().index[len(raw) // 2 + 7]], 260)
display(HTML("<div style='display:grid;grid-template-columns:1fr 1fr;gap:20px'><div><b>Chotu (SmolLM2)</b>" + X.token_lens(tok, text) +
             "</div><div><b>GPT-2</b>" + X.token_lens(gpt2, text) + '</div></div>'))

**Think about it**
1. The two tokenizers are close on prose, yet Chotu's splits every number into digits. Where must Chotu be *saving* tokens to end up nearly even overall?
2. We'll compare Chotu with GPT-2 on HellaSwag. Why is that fair even though the tokenizers differ? (Hint: what does the benchmark score?)

## 12 · Lossless, and what's actually in the shards

A good tokenizer is **lossless**: decoding the ids gives back exactly the original text, byte for byte. We check this on all 14,000 documents. Then we look directly at our training shard, which is nothing but a long list of 16-bit numbers. Red marks show where one document ends and the next begins.

In [ ]:
dec = tok.batch_decode(enc, clean_up_tokenization_spaces=False)
same = np.array([d == t for d, t in zip(dec, raw.text)])
print(f'exact round trip on {same.mean():.2%} of {len(raw):,} documents ({(~same).sum()} differ)')
arr = np.memmap(train_path, dtype=np.uint16, mode='r')
print(f'{train_path.name}: {len(arr):,} tokens × 2 bytes = {arr.nbytes / 1e9:.2f} GB | largest id {int(arr[:50_000_000].max()):,} fits in 16 bits (max 65,535)')
display(HTML(X.shard_window(tok, train_path, n=140, seed=3)))

**Think about it**
1. Why is "lossless" essential for a chat model that might be asked to copy a quote or some code?
2. The shard stores ids as 16-bit numbers. What would break if the vocabulary had 70,000 tokens?

## 13 · What a chat looks like to the model

After pretraining, chat tuning wraps every turn in **ChatML** markers: `<|im_start|>`, a role, the text, `<|im_end|>`. The special markers are single tokens (ids 1 and 2) that never appeared in pretraining. Below is one of *our* test prompts in that format, then a real conversation from `smol-smoltalk` (our chat-tuning data). **Red** tokens are the ones the model is trained to predict (assistant replies plus their `<|im_end|>`). Everything else is context only.

In [ ]:
prompts = re.findall(r'^\d+\.\s+(.+)$', (REPO / 'prompts' / 'test_prompts.md').read_text(), flags=re.M)
chatml = lambda msgs: ''.join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in msgs)
p_text = chatml([{'role': 'user', 'content': prompts[0]}]) + '<|im_start|>assistant\n'
p_ids = tok(p_text, add_special_tokens=False)['input_ids']
display(HTML(chips(p_ids, note=f'<b>Test prompt 1, ready for the model to answer</b> ({len(p_ids)} tokens)', mark=lambda k, i: i in (1, 2))))

files = sorted(fs.glob('datasets/HuggingFaceTB/smol-smoltalk/data/train-*.parquet'))
pf = pq.ParquetFile(fs.open(files[0], 'rb'))
conv = next(m for m in pf.read_row_group(0, columns=['messages']).column('messages').to_pylist() if 2 <= len(m) <= 4 and sum(len(x['content']) for x in m) < 900)
ids_, labels = [], []
for m in conv:
    head = tok(f"<|im_start|>{m['role']}\n", add_special_tokens=False)['input_ids']
    body = tok(m['content'] + '<|im_end|>', add_special_tokens=False)['input_ids']
    nl = tok('\n', add_special_tokens=False)['input_ids']
    ids_ += head + body + nl
    labels += [False] * len(head) + [m['role'] == 'assistant'] * len(body) + [False] * len(nl)
display(HTML(chips(ids_, note=f'<b>A real smol-smoltalk conversation</b>: {len(ids_)} tokens, {sum(labels)} of them trained on (red)', mark=lambda k, i: labels[k])))
try:
    raw_ids = tok('<|im_start|>', add_special_tokens=False, split_special_tokens=True)['input_ids']
    display(HTML(chips(raw_ids, note='The same text <code>&lt;|im_start|&gt;</code> treated as plain characters instead of a marker')))
except TypeError:
    pass

**Think about it**
1. Why is the model trained only on the red tokens, not on the user's message?
2. The `<|im_end|>` token is red: the model learns to predict it. What would happen at chat time if it never learned that?
3. If a user types the literal text `<|im_start|>` into a message, should it become marker id 1 or plain characters? What could go wrong with each choice?

## 14 · What this means for Chotu

In [ ]:
emb = N * 640
rows = {
    'vocabulary': f'{N:,} ids ({(counts > 0).sum():,} used in our data, {len(never):,} never used)',
    'bytes per token (bits-per-byte conversion)': f'{raw.n_byte.sum() / raw.n_tok.sum():.2f}',
    'characters in a 2,048-token window': f'≈{2048 * raw.n_byte.sum() / raw.n_tok.sum():,.0f} bytes, about {2048 * raw.n_byte.sum() / raw.n_tok.sum() / 6:,.0f} words',
    'tokens per word (as words appear)': f'{np.average(alpha.tokens, weights=alpha.freq):.2f}',
    'digits': 'always one token each',
    'embedding table (shared with output layer)': f'{N:,} × 640 = {emb/1e6:.1f}M parameters ({emb / 180_925_056:.0%} of the model)',
    'round trip': f'{same.mean():.2%} of documents decode back exactly',
    'vs GPT-2': f'{raw.n_tok.sum() / raw.token_count.sum():.3f}× as many tokens',
}
display(pd.DataFrame({'value': rows}))

## Summary: what you should now know

- **What it is:** a byte-level BPE with 49,152 ids, built from 17 special tokens, 235 single-byte tokens and 48,900 learned merges. It never changes your text (no normalizer), and it has 17 special tokens, of which we use three.
- **How it looks:** common words are one token *with their leading space* (`␣the`). Rare words, names, URLs, code and other languages break into several pieces. Every digit is its own token.
- **Vocabulary vs usage:** a steep long tail. A thousand tokens cover most text, and a few hundred ids are never used in 300M tokens, mostly code and non-English pieces from SmolLM2's training mix.
- **Compression:** about 4.5 bytes per token on our data, nearly the same as GPT-2. It's much worse on tables, maths and non-English text, which is why we report bits per byte.
- **Chat:** `<|im_start|>` and `<|im_end|>` are single tokens the model first meets during chat tuning, where only the assistant's tokens are trained on.

**Next:** answer the questions in `docs/LEARNING.md`. If something surprised you, add it to `docs/PROGRESS.md` under *Open questions*.